## NLP Feature Extraction Pipeline
This notebook loads the raw, uncleaned scraped data and processes the text fields to generate lightweight NLP features for the Streamlit dashboard.

In [1]:
!pip install textstat
!pip install vaderSentiment

In [2]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from transformers import pipeline
import textstat
import sys
import os
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sys.path.append(os.path.abspath(".."))
from processing.functions import *

data_path = "../data/cars_and_bids_full_history_v3.csv"
cols_to_keep = ['URL', 'Make', 'Model', 'Sold_Price', 'Modifications', 'Equipment', 'Highlights', 'Known Flaws', 'Recent Service History', 'Ownership History', 'Other Items Included in Sale', 'Seller Notes']
df = pd.read_csv(data_path, usecols=cols_to_keep)

# Quick cleaning of Model and Sold_Price columns
df['Sold_Price'] = df['Sold_Price'].apply(clean_currency)
df = df.dropna(subset=['Sold_Price'])
df['Model'] = df['Model'].apply(clean_model)

pd.set_option('display.max_columns', None)

df.head()

2026-03-14 21:48:25.562999: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


,URL,Sold_Price,Make,Model,Highlights,Equipment,Modifications,Known Flaws,Recent Service History,Ownership History,Other Items Included in Sale,Seller Notes
0,https://carsandbids.com/auctions/rkdqRPJN/2026...,226000.0,Mercedes-Benz,G Wagen,"THIS… is a 2026 Mercedes-AMG G63, finished in ...",A build sheet is provided in the photo gallery...,NaN,NaN,The seller states that this G63 has not requir...,The seller is representing this G63 on behalf ...,2 keys Owner's manuals,"There is a loan on this vehicle, and sale proc..."
1,https://carsandbids.com/auctions/30Op4L4g/2026...,76500.0,BMW,G8X M4,"THIS... is a 2026 BMW M4 Coupe, finished in Fr...","A build sheet is provided in the gallery, and ...",NaN,NaN,The attached Carfax history report shows that ...,The seller reportedly purchased this M4 new in...,2 keys Owner's manual Window sticker,The seller states that clear paint protection ...
2,https://carsandbids.com/auctions/3010V6BV/2026...,215485.0,Mercedes-Benz,G Wagen,"THIS… is a 2026 Mercedes-AMG G63, finished in ...",A build sheet is provided in the photo gallery...,NaN,NaN,The seller states that this G63 has not requir...,The seller reportedly purchased this G63 new i...,2 keys Owner's manual Build sheet All-weather ...,"There is a loan on this vehicle, and sale proc..."
3,https://carsandbids.com/auctions/KZB8oVXk/2026...,356000.0,Porsche,992 911,"THIS... is a 2026 Porsche 911 GT3 Touring, fin...",A build sheet is provided in the photo gallery...,NaN,NaN,The selling dealer states that this 911 has no...,The selling dealer reportedly acquired this 91...,2 keys Owner's manual Build sheet,NaN
7,https://carsandbids.com/auctions/9lDZOmLz/2026...,66000.0,Rivian,R1S,"THIS… is a 2026 Rivian R1S Adventure Edition, ...",A partial list of notable factory equipment re...,NaN,NaN,The seller states that this Rivian has not req...,The seller reportedly purchased this Rivian ne...,2 key cards Safety Guide All-weather floor mat...,"There is a loan on this vehicle, and sale proc..."


### 1. Entity Recognition: Premium Aftermarket Brands
Extracts specific high-value brands from the `Modifications` column.


In [3]:
brands = [
    "bilstein", "ohlins", "bbs", "recaro", "dinan", 
    "kw", "michelin", "koni", "brembo", "borla", "bosch",
    "apr", "momo", "nardi", "holley", "edgemotorsport", "moog", "acdelco", "hks"
]

def extract_brands(text):
    if pd.isna(text) or not isinstance(text, str):
        return []

    text_lower = text.lower()
    found_brands = []

    for brand in brands:
        if re.search(rf'\b{brand}\b', text_lower):
            found_brands.append(brand.capitalize())

    return found_brands

# Combine relevant fields
df['Combined_Mods'] = df['Modifications'].fillna("") + " " + \
                      df['Equipment'].fillna("") + " " + \
                      df.get('Other Items Included in Sale', pd.Series("")).fillna("")
                      
df['Extracted_Brands'] = df['Combined_Mods'].apply(extract_brands)
df['Has_Premium_Mods'] = df['Extracted_Brands'].apply(lambda x: 1 if len(x) > 0 else 0)
df['Premium_Brand_Count'] = df['Extracted_Brands'].apply(len)
df['Extracted_Brands_List'] = df['Extracted_Brands'].apply(lambda x: ", ".join(x))

# Preview listings that actually have premium mods
display(df[df['Has_Premium_Mods'] == 1][['Make', 'Model', 'Modifications', 'Equipment', 'Other Items Included in Sale', 'Extracted_Brands_List']].head())


,Make,Model,Modifications,Equipment,Other Items Included in Sale,Extracted_Brands_List
34,Subaru,BRZ,NaN,A partial list of notable equipment reported b...,2 keys Owner's manual Rubber floor and trunk mats,Brembo
37,Lexus,IS,NaN,A build sheet is provided in the photo gallery...,2 keys 2 key holders Owner's manual Lexus of Q...,Bbs
39,Toyota,Tacoma,NaN,A build sheet is provided in the photo gallery...,2 keys Owner's manual Booklets,Bilstein
41,BMW,G80 M3,Notable modifications reported by the seller i...,A window sticker is provided in the photo gall...,2 keys Owner's manual Window sticker All-weath...,Michelin
45,Subaru,BRZ,Notable modifications reported by the seller i...,A Monroney Label is provided in the photo gall...,2 key fobs Window sticker Set of winter tires ...,Brembo


### 2. Topic Modeling: Listing Archetypes
Uses Non-Negative Matrix Factorization (NMF) to cluster cars based on the combined text of their Highlights and Recent Service history.

In [8]:
# text_data = df['Highlights'].fillna("") + " " + \
#                 df['Modifications'].fillna("") + " " + \
#                 df['Known Flaws'].fillna("")

text_data = df['Known Flaws'].fillna("")
text_list = text_data.tolist()

# Vectorize
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(text_list)

# Fit NMF (adjust n_components to find the best archetypes)
N_TOPICS = 4
nmf_model = NMF(n_components=N_TOPICS, random_state=42, init='nndsvda')
nmf_features = nmf_model.fit_transform(tfidf_matrix)

df['Archetype_Cluster'] = nmf_features.argmax(axis=1)

# Display the keywords for each cluster so you can manually interpret them
feature_names = tfidf_vectorizer.get_feature_names_out()
print("\n--- Cluster Keywords ---")
for topic_idx, topic in enumerate(nmf_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-10 - 1:-1]]
    print(f"Cluster {topic_idx}: {', '.join(top_words)}")
    
# Preview
display(df[['Make', 'Model', 'Highlights', 'Archetype_Cluster']].head())


--- Cluster Keywords ---
Cluster 0: exterior, wear, interior, seats, touch, points, scratches, chips, tires, date
Cluster 1: damage, report, minor, carfax, attached, history, sustained, indicates, rear, vehicle
Cluster 2: driver, seat, bumper, rear, passenger, door, scratches, chips, creases, gallery
Cluster 3: pictured, gallery, squad, lemon, inspection, purchase, pre, listing, additionally, included


,Make,Model,Highlights,Archetype_Cluster
0,Mercedes-Benz,G Wagen,"THIS… is a 2026 Mercedes-AMG G63, finished in ...",0
1,BMW,G8X M4,"THIS... is a 2026 BMW M4 Coupe, finished in Fr...",0
2,Mercedes-Benz,G Wagen,"THIS… is a 2026 Mercedes-AMG G63, finished in ...",0
3,Porsche,992 911,"THIS... is a 2026 Porsche 911 GT3 Touring, fin...",0
7,Rivian,R1S,"THIS… is a 2026 Rivian R1S Adventure Edition, ...",0


## 3. Seller Persona: Zero-Shot Classification
Uses a Hugging Face transformer model to categorize the seller's tone.

In [ ]:
# Initialize the pipeline (downloads model weights on first run)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

PERSONA_LABELS = [
    "Passionate Enthusiast", 
    "Clinical Dealership", 
    "Reluctant Seller",
    "Flipping for Profit"
]

def get_seller_persona(text):
    if pd.isna(text) or not isinstance(text, str) or len(text.strip()) < 10:
        return "Unknown"
        
    # Truncate to avoid exceeding max token lengths
    truncated_text = text[:1000] 
    result = classifier(truncated_text, PERSONA_LABELS)
    return result['labels'][0]

if 'Highlights' in df.columns:
    # OPTIONAL: Test on a small sample first before running on the whole dataset
    sample_size = 5
    print(f"Testing zero-shot classification on a sample of {sample_size} rows...")
    
    sample_df = df.head(sample_size).copy()
    sample_df['Seller_Persona'] = sample_df['Highlights'].apply(get_seller_persona)
    display(sample_df[['Make', 'Model', 'Highlights', 'Seller_Persona']])
    
    # UNCOMMENT the lines below to run on the entire dataset
    # print("Running classification on the full dataset...")
    # df['Seller_Persona'] = df['Highlights'].apply(get_seller_persona)
else:
    print("Column 'Highlights' not found.")

## 4. Seller Transparency and "Effort" Scoring
Calculates word count, structural density (bullet points), readability, and sentiment across the main text fields to gauge seller effort.

In [10]:
analyzer = SentimentIntensityAnalyzer()

def calculate_effort_metrics(text):
    if pd.isna(text) or not isinstance(text, str) or len(text.strip()) == 0:
        return 0, 0, 0, 0
    
    # 1. Length/Detail Volume
    word_count = textstat.lexicon_count(text, removepunct=True)
    
    # 2. Readability (Flesch Reading Ease: higher is easier to read)
    readability = textstat.flesch_reading_ease(text)
    
    # 3. Structural Density (Bullet points imply organized data)
    bullet_points = text.count('- ') + text.count('* ') + text.count('•')
    
    # 4. Sentiment (Compound score from -1 to 1)
    sentiment_dict = analyzer.polarity_scores(text)
    compound_sentiment = sentiment_dict['compound']
    
    return word_count, readability, bullet_points, compound_sentiment

# Combine the main descriptive text fields to evaluate total effort
text_cols_to_check = ['Highlights', 'Known Flaws', 'Equipment', 'Modifications', 'Seller Notes', 'Recent Service History']

if all(col in df.columns for col in text_cols_to_check):
    print("Calculating effort metrics...")

    # Combine everything for a total footprint of the seller's effort
    df['Combined_Seller_Text'] = df['Highlights'].fillna("") + "\n" + \
                                 df['Known Flaws'].fillna("") + "\n" + \
                                 df['Equipment'].fillna("") + "\n" + \
                                 df['Modifications'].fillna("") + "\n" + \
                                 df['Recent Service History'].fillna("") + "\n" + \
                                 df['Seller Notes'].fillna("")

    # Apply the scoring function
    metrics = df['Combined_Seller_Text'].apply(calculate_effort_metrics)

    # Expand the returned tuples into distinct columns
    df[['Word_Count', 'Readability_Score', 'Bullet_Point_Count', 'Sentiment_Score']] = pd.DataFrame(metrics.tolist(), index=df.index)

    # Create a simple composite score (adjust weights as you see fit)
    # E.g., 1 point per word, 10 points per organized bullet point
    df['Composite_Effort_Score'] = df['Word_Count'] + (df['Bullet_Point_Count'] * 10)
    
    # Preview the results
    display(df[['Make', 'Model', 'Word_Count', 'Readability_Score', 'Composite_Effort_Score', 'Sentiment_Score']].head())
else:
    print(f"Missing one of the required text columns: {text_cols_to_check}")

Calculating effort metrics...


,Make,Model,Word_Count,Readability_Score,Composite_Effort_Score,Sentiment_Score
0,Mercedes-Benz,G Wagen,313,40.349529,313,0.9689
1,BMW,G8X M4,346,41.607304,346,0.9673
2,Mercedes-Benz,G Wagen,350,38.848238,350,0.9711
3,Porsche,992 911,304,38.667379,304,0.8015
7,Rivian,R1S,309,48.759010,309,0.9308


## Export Results
Drops the massive raw text columns and saves the lightweight NLP features into the frontend data directory for quick loading in Streamlit.


In [ ]:
OUTPUT_BRANDS = "../data/frontend_data/nlp_brands.csv"
OUTPUT_ARCHETYPES = "../data/frontend_data/nlp_archetypes.csv"
OUTPUT_PERSONAS = "../data/frontend_data/nlp_personas.csv"
OUTPUT_EFFORT = "../data/frontend_data/nlp_effort_scores.csv"

# 1. Export Brands
brands_cols = ['URL', 'Make', 'Model', 'Sold_Price', 'Has_Premium_Mods', 'Premium_Brand_Count', 'Extracted_Brands_List']
if set(brands_cols).issubset(df.columns):
    df[brands_cols].to_csv(OUTPUT_BRANDS, index=False)
    print(f"Saved Brands to {OUTPUT_BRANDS}")

# 2. Export Archetypes
archetypes_cols = ['URL', 'Make', 'Model', 'Sold_Price', 'Archetype_Cluster']
if set(archetypes_cols).issubset(df.columns):
    df[archetypes_cols].to_csv(OUTPUT_ARCHETYPES, index=False)
    print(f"Saved Archetypes to {OUTPUT_ARCHETYPES}")

# 3. Export Personas (Only if you ran the full dataset in Cell 4)
personas_cols = ['URL', 'Make', 'Model', 'Sold_Price', 'Seller_Persona']
if set(personas_cols).issubset(df.columns):
    df[personas_cols].to_csv(OUTPUT_PERSONAS, index=False)
    print(f"Saved Personas to {OUTPUT_PERSONAS}")

# 4. Export Effort Scores
effort_cols = [
    'URL', 'Make', 'Model', 'Sold_Price', 
    'Word_Count', 'Readability_Score', 'Bullet_Point_Count', 
    'Sentiment_Score', 'Composite_Effort_Score'
]

if set(effort_cols).issubset(df.columns):
    df[effort_cols].to_csv(OUTPUT_EFFORT, index=False)
    print(f"Saved Effort Scores to {OUTPUT_EFFORT}")